# MultiMend — Giai đoạn 1: so sánh B0 và B1 trên QuixBugs-Python

Notebook này chạy trên Kaggle GPU và giữ nguyên pipeline chính thức:

 \`buggy/correct programs → bug-line/context extraction → CodeT5 checkpoint ensemble → combine → pytest validation\`

- **B0:** không dùng RAG (\`no_rag\`).
- **B1:** dùng local RAG top-5, cosine distance \`<= 0.5\` (\`fixed_rag\`).
- Mỗi chiến lược dùng cùng 5 checkpoint cuối, cùng seed và cùng 100 candidates/hunk/checkpoint.
- Có thể chạy lại từng chiến lược; output của chiến lược đã hoàn tất sẽ được giữ lại.


In [ ]:
# Cell 1 — Kaggle setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ThaoTo2806/MultiMend28.git"
REPO = Path("/kaggle/working/MultiMend28")

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Repository:", REPO)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("Torch will be checked after dependency installation.")


In [ ]:
# Cell 2 — Install the repository dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "-r", "requirements.txt", "more-itertools", "pygments",
    ],
    check=True,
)
print("Dependencies installed.")


## Cell 3 — Locate the benchmark and inference checkpoints

If the checkpoints are attached as a Kaggle Dataset, set \`CHECKPOINT_SOURCE\` to
that mounted directory. The directory must contain \`checkpoint-*\` folders with
\`trainer_state.json\`, model files and tokenizer files. The cell copies them to
\`models/multimend-codet5-small\`, which is the path used by the official script.


In [ ]:
# Cell 3 — Prepare benchmark data and checkpoints
import json
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")
os.chdir(REPO)

# Change this to the mounted Kaggle Dataset path when checkpoints are not in the repo.
CHECKPOINT_SOURCE = Path(os.environ.get(
    "MULTIMEND_CHECKPOINT_SOURCE",
    "/kaggle/input/multimend-checkpoints/models/multimend-codet5-small",
))
MODEL_DIR = REPO / "models" / "multimend-codet5-small"
QB_DIR = REPO / "benchmarks" / "QuixBugs"

if not (QB_DIR / "python_programs").is_dir():
    raise FileNotFoundError(f"QuixBugs benchmark is missing: {QB_DIR}")
if not (QB_DIR / "correct_python_programs").is_dir():
    raise FileNotFoundError(f"Correct QuixBugs programs are missing: {QB_DIR}")

checkpoint_dirs = sorted(
    p for p in CHECKPOINT_SOURCE.glob("checkpoint-*")
    if (p / "trainer_state.json").is_file()
)
if checkpoint_dirs:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for checkpoint in checkpoint_dirs:
        destination = MODEL_DIR / checkpoint.name
        if not destination.exists():
            shutil.copytree(checkpoint, destination)
    print(f"Copied {len(checkpoint_dirs)} checkpoints from {CHECKPOINT_SOURCE}")
elif not list(MODEL_DIR.glob("checkpoint-*")):
    raise FileNotFoundError(
        "No checkpoints found. Attach the checkpoint Kaggle Dataset and set "
        "MULTIMEND_CHECKPOINT_SOURCE to its model directory."
    )

print("Model directory:", MODEL_DIR)
print("Checkpoints:", [p.name for p in sorted(MODEL_DIR.glob("checkpoint-*"))])


In [ ]:
# Cell 4 — Build official QuixBugs-Python metadata
from src.bugline_finders import quixbugs_python

metadata_file = REPO / "outputs" / "QuixBugs-Python" / "QuixBugs_Python.jsonl"
metadata_file.parent.mkdir(parents=True, exist_ok=True)

if not metadata_file.exists():
    quixbugs_python.main()

required_metadata = [
    metadata_file,
    metadata_file.parent / "rem.txt",
    metadata_file.parent / "add.txt",
]
missing = [str(path) for path in required_metadata if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Metadata generation failed: {missing}")

bug_count = sum(1 for _ in metadata_file.open(encoding="utf-8"))
print("Metadata:", metadata_file)
print("Bug records:", bug_count)


## Cell 5 — Run B0 and B1 generation

The official generator selects the five checkpoints with the largest
 \`trainer_state.json:global_step\`. \`MULTIMEND_CONTEXT_STRATEGY\` changes only the
context construction; all generation parameters remain identical.

The two subprocesses are independent and write to:

- \`outputs/QuixBugs-Python/outputs-multimend-no_rag\`
- \`outputs/QuixBugs-Python/outputs-multimend-fixed_rag\`

If a strategy already has a complete \`sequences_100.jsonl\`, it is skipped so a
Kaggle session can resume without overwriting completed work.


In [ ]:
# Cell 5 - Generate both experimental conditions
import os
import subprocess
import sys

output_root = REPO / "outputs" / "QuixBugs-Python"
generator_path = REPO / "src" / "generate_primary_candidates.py"
if not generator_path.is_file():
    raise FileNotFoundError(f"Missing generator source: {generator_path}")

source = generator_path.read_text(encoding="utf-8")
required_markers = (
    'context_strategy = os.environ.get("MULTIMEND_CONTEXT_STRATEGY", "fixed_rag")',
    "decoder_start_token_id = next(",
)
missing_markers = [marker for marker in required_markers if marker not in source]
if missing_markers:
    raise RuntimeError(
        "The checked-out Python source is incomplete. "
        f"Missing markers: {missing_markers}. "
        "Push the updated src files and clone that revision on Kaggle."
    )

def run_generation(strategy: str) -> None:
    strategy_dir = output_root / f"outputs-multimend-{strategy}"
    raw_file = strategy_dir / "sequences_100.jsonl"
    if raw_file.is_file() and raw_file.stat().st_size > 0:
        print(f"{strategy}: existing output found, skipping generation")
        return
    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy
    env["PYTHONPATH"] = str(REPO)
    print(f"Starting {strategy}...")
    subprocess.run(
        [sys.executable, "-m", "src.generate_primary_candidates"],
        cwd=REPO,
        env=env,
        check=True,
    )
    if not raw_file.is_file():
        raise RuntimeError(f"{strategy}: generator did not create {raw_file}")
    print(f"{strategy}: generation complete ({raw_file.stat().st_size:,} bytes)")

for strategy in ("no_rag", "fixed_rag"):
    run_generation(strategy)



In [ ]:
# Cell 6 — Combine and deduplicate candidates for B0/B1
def run_combine(strategy):
    final_file = (
        REPO / "outputs" / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}" / "final_candidates_100.jsonl"
    )
    if final_file.is_file() and final_file.stat().st_size > 0:
        print(f"{strategy}: existing combined output found, skipping")
        return
    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy
    subprocess.run(
        [sys.executable, "-m", "src.combine_checkpoints_results"],
        cwd=REPO,
        env=env,
        check=True,
    )
    if not final_file.is_file():
        raise RuntimeError(f"{strategy}: combine did not create {final_file}")
    print(f"{strategy}: combine complete")

for strategy in ("no_rag", "fixed_rag"):
    run_combine(strategy)


In [ ]:
# Cell 7 — Validate candidates with the official QuixBugs pytest validator
def run_validation(strategy):
    result_file = (
        REPO / "outputs" / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}" / "plausible_candidates_100.jsonl"
    )
    if result_file.is_file() and result_file.stat().st_size > 0:
        print(f"{strategy}: existing validation output found, skipping")
        return
    env = os.environ.copy()
    env["MULTIMEND_CONTEXT_STRATEGY"] = strategy
    subprocess.run(
        [sys.executable, "-m", "src.validators.validate_quixbugs_python"],
        cwd=REPO,
        env=env,
        check=True,
    )
    if not result_file.is_file():
        raise RuntimeError(f"{strategy}: validator did not create {result_file}")
    print(f"{strategy}: validation complete")

for strategy in ("no_rag", "fixed_rag"):
    run_validation(strategy)


## Cell 8 — Metrics and B0/B1 comparison

Metrics are computed from the official validator output:

- plausible bugs: at least one plausible patch for every hunk;
- exact-match bugs: at least one candidate equal to the reference patch;
- correct bugs: the official validator's \`correct\` flag, when available;
- context instrumentation: retrieved count and truncation statistics from B1.


In [ ]:
# Cell 8 — Produce a machine-readable comparison
import json
import pandas as pd

def load_jsonl(path):
    return pd.read_json(path, orient="records", lines=True)

def summarize(strategy):
    base = (
        REPO / "outputs" / "QuixBugs-Python"
        / f"outputs-multimend-{strategy}"
    )
    candidates = load_jsonl(base / "final_candidates_100.jsonl")
    validated = load_jsonl(base / "plausible_candidates_100.jsonl")

    grouped = validated.groupby(["bugid", "hunk"], dropna=False)
    plausible_hunks = grouped["plausible"].any()
    plausible_bugs = plausible_hunks.groupby(level=0).all()
    exact_bugs = (
        grouped["exact_match"].any().groupby(level=0).any()
        if "exact_match" in validated
        else pd.Series(dtype=bool)
    )
    correct_bugs = (
        grouped["correct"].any().groupby(level=0).any()
        if "correct" in validated
        else pd.Series(dtype=bool)
    )

    context_log = base / f"context_log_{strategy}.jsonl"
    context = load_jsonl(context_log) if context_log.exists() else pd.DataFrame()
    return {
        "strategy": strategy,
        "bugs": int(validated["bugid"].nunique()),
        "hunks": int(validated[["bugid", "hunk"]].drop_duplicates().shape[0]),
        "raw_candidates": int(
            load_jsonl(base / "sequences_100.jsonl").shape[0]
        ),
        "final_candidates": int(candidates.shape[0]),
        "plausible_bugs": int(plausible_bugs.sum()),
        "exact_match_bugs": int(exact_bugs.sum()),
        "correct_bugs": int(correct_bugs.sum()),
        "retrieved_contexts": (
            int((context["retrieved_count"] > 0).sum())
            if not context.empty else 0
        ),
        "truncated_inputs": (
            int(context["was_truncated"].sum())
            if not context.empty else 0
        ),
        "local_context_at_risk": (
            int(context["local_context_at_risk"].sum())
            if not context.empty else 0
        ),
    }

summary = pd.DataFrame([summarize("no_rag"), summarize("fixed_rag")])
summary["plausible_rate"] = summary["plausible_bugs"] / summary["bugs"]
summary["exact_match_rate"] = summary["exact_match_bugs"] / summary["bugs"]
summary["correct_rate"] = summary["correct_bugs"] / summary["bugs"]

summary_path = output_root / "stage1_b0_b1_comparison.json"
summary.to_json(summary_path, orient="records", indent=2)
display(summary)
print("Saved:", summary_path)


In [ ]:
# Cell 9 — Final reproducibility checks
import json

for strategy in ("no_rag", "fixed_rag"):
    directory = output_root / f"outputs-multimend-{strategy}"
    required = [
        directory / "generated_input.jsonl",
        directory / "sequences_100.jsonl",
        directory / "final_candidates_100.jsonl",
        directory / "plausible_candidates_100.jsonl",
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"{strategy} incomplete: {missing}")

manifest = {
    "stage": 1,
    "dataset": "QuixBugs-Python",
    "strategies": ["no_rag", "fixed_rag"],
    "seed": 42,
    "num_checkpoints": 5,
    "num_return_sequences": 100,
    "rag": {"top_k": 5, "distance_threshold": 0.5},
    "comparison_file": str(summary_path),
}
(output_root / "stage1_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, indent=2))
print("STAGE 1 — B0/B1 COMPLETE")
